In [2]:
!git config --global user.name "<username>"
!git config --global user.email "<email>"

!git clone https://<token>@github.com/ThreadZepplin/llm-training
%cd llm-training

Cloning into 'llm-training'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 110 (delta 40), reused 81 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 50.44 KiB | 1.94 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/kaggle/working/llm-training


In [3]:
!git pull
!git switch training-olmo
!git branch

Already up to date.
Branch 'training-olmo' set up to track remote branch 'training-olmo' from 'origin'.
Switched to a new branch 'training-olmo'
  main
* training-olmo


In [4]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.6 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 16.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 115.2 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [jupyter]m1/3 [trl]


In [5]:
from pathlib import Path
import shutil

repo_root = Path("/kaggle/working/llm-training")
raw_dir = repo_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

copied = []

for dataset_dir in Path("/kaggle/input").iterdir():
    if dataset_dir.is_dir():
        print("Checking dataset folder:", dataset_dir)
        for path in dataset_dir.rglob("*"):
            if path.is_file() and path.suffix.lower() == ".csv":
                shutil.copy(path, raw_dir / path.name)
                copied.append(path.name)

print("Copied files to:", raw_dir)
print("Copied", len(copied), "CSV files")
print(copied)

Checking dataset folder: /kaggle/input/datasets
Copied files to: /kaggle/working/llm-training/data/raw
Copied 25 CSV files
['7.5in x 8in Hex1 2xBrim Front Right-20250925083521.csv', '7.5in x 8in Hex1 2xBrim Front Right-20250926080727.csv', '7.5in x 8in Hex1 2xBrim Front Right-20251007080848.csv', '7.5in x 8in Hex2 2xBrim Front Left-20250916111958.csv', '017_Sample_033_ros.csv', '7.5in x 8in Hex 2xBrim Rear Right PolyPro-20250924083722.csv', '7.5in x 8in Hex1 2xBrim Front Right-20250915080848.csv', '7.5in x 8in Hex1 2xBrim Front Right-20251002084414.csv', '7.5in x 8in Hex 2xBrim Rear Left PolyPro-20251003080433.csv', '7.5in x 8in Hex1 2xBrim Front Right-20250916080815.csv', '7.5in x 8in Hex 2xBrim Rear Right PolyPro-20251001130529.csv', '7.5in x 8in Hex1 2xBrim Front Right-20250930081421.csv', '7.5in x 8in Hex1 2xBrim Front Right-20250929081954.csv', '7.5in x 8in Hex 2xBrim Rear Left PolyPro-20250924115019.csv', '015_Sample_031_ros.csv', '7.5in x 8in Hex2 2xBrim Front Left-2025091512413

In [6]:
!python scripts/preprocess_data.py --input-dir data/raw --output-file data/processed/operations.jsonl

Wrote 25 records to data/processed/operations.jsonl


In [7]:
!python scripts/generate_synthetic_records.py --input-file data/processed/operations.jsonl --output-file data/processed/synthetic_operator_records.jsonl

Wrote 21 synthetic operator records to data/processed/synthetic_operator_records.jsonl


In [8]:
!python scripts/build_dataset_splits.py --real-file data/processed/operations.jsonl --synthetic-file data/processed/synthetic_operator_records.jsonl --out-all data/processed/dataset_all.jsonl --out-train data/processed/dataset_train.jsonl --out-test data/processed/dataset_test_manual.jsonl --out-meta data/processed/dataset_split_metadata.json

Wrote 46 total examples to data/processed/dataset_all.jsonl
Wrote 36 train examples to data/processed/dataset_train.jsonl
Wrote 10 test examples to data/processed/dataset_test_manual.jsonl
Wrote split metadata to data/processed/dataset_split_metadata.json
Held-out groups:
  - 015_Sample_023_ros.csv
  - 015_Sample_031_ros.csv
  - 7.5in x 8in Hex 2xBrim Rear Left PolyPro-20250924115019.csv
  - 7.5in x 8in Hex 2xBrim Rear Left PolyPro-20251003080433.csv
  - 7.5in x 8in Hex 2xBrim Rear Left-20251010095359.csv
  - 7.5in x 8in Hex 2xBrim Rear Right PolyPro-20251001130529.csv


In [9]:
!python scripts/prepare_finetune_files.py --train-input data/processed/dataset_train.jsonl --test-input data/processed/dataset_test_manual.jsonl --out-train data/processed/finetune_train.jsonl --out-test-inputs data/processed/finetune_test_inputs.jsonl --out-test-gold data/processed/finetune_test_gold.jsonl

Wrote 36 training rows to data/processed/finetune_train.jsonl
Wrote 10 test input rows to data/processed/finetune_test_inputs.jsonl
Wrote 10 test gold rows to data/processed/finetune_test_gold.jsonl


In [10]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 19.9 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 16.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 46.9 MB/s  0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1━━━━━━━━━━━━ 1/3 [huggingface-hub]
    Uninstalling huggingface_hub-1.4.1:━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [huggingface-hub]
      Successfully uninstalled huggingface_hub-1.4.1━━━━━━━━━━ 1/3 [huggingface-hub]
  Attempting uninstall: transformers━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [huggingface-hub]
    Found existing installation: transformers 5.0.00m━━━━━━━━━━━━━ 2/3 [transformers]
    Uninstalling transformers-5.0.0:0m╸━━━━━━━━━━━━━ 2/3 [transformers]
      Successfully uninstalled transformers-5.0.0━━━━━━━━━━━━━ 2/3 

In [11]:
!git pull

Already up to date.


In [12]:
!python scripts/train_olmo_lora.py --config configs/olmo_lora.json

config.json: 1.57kB [00:00, 844kB/s]
tokenizer_config.json: 4.31kB [00:00, 10.6MB/s]
tokenizer.json: 7.14MB [00:00, 19.8MB/s]
Generating train split: 36 examples [00:00, 2726.62 examples/s]
Map: 100%|██████████████████████████████| 36/36 [00:00<00:00, 517.38 examples/s]
model.safetensors: 100%|████████████████████| 14.9G/14.9G [00:59<00:00, 249MB/s]
The fast path is not available because one of the required libraries is not installed. Falling back to torch implementation. To install, follow: https://github.com/fla-org/flash-linear-attention#installation
  0%|                                                    | 0/15 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/models/olmo_hybrid/modeling_olmo_hybrid.py:983: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  causal_mask = create_causal_mask(
{'loss': '2.329', 'grad_norm': '0.06156', 'learning_rate': '0.0002', 'epoch': '0.2222'}

In [23]:
!git pull

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 391 bytes | 391.00 KiB/s, done.
From https://github.com/ThreadZepplin/llm-training
   77eb6a3..b5bbe90  training-olmo -> origin/training-olmo
Updating 77eb6a3..b5bbe90
Fast-forward
 src/modeling/olmo_inference.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [24]:
!python scripts/eval_olmo_lora.py \
  --base-model allenai/Olmo-Hybrid-7B \
  --adapter-path outputs/olmo_lora_manufacturing \
  --test-inputs data/processed/finetune_test_inputs.jsonl \
  --output-file outputs/olmo_test_predictions.jsonl

`torch_dtype` is deprecated! Use `dtype` instead!
The fast path is not available because one of the required libraries is not installed. Falling back to torch implementation. To install, follow: https://github.com/fla-org/flash-linear-attention#installation
Loading weights: 100%|████████████████████████| 523/523 [00:06<00:00, 86.10it/s]
/usr/local/lib/python3.12/dist-packages/transformers/models/olmo_hybrid/modeling_olmo_hybrid.py:983: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  causal_mask = create_causal_mask(
ex_0005: 66.66s
ex_0011: 66.23s
ex_0017: 65.79s
ex_0019: 66.13s
ex_0023: 67.90s
ex_0024: 68.69s
ex_0030: 68.24s
ex_0036: 68.53s
ex_0044: 67.68s
ex_0045: 65.78s

Wrote predictions to outputs/olmo_test_predictions.jsonl
Total inference time: 671.62s
Average time per example: 67.16s


In [25]:
!python scripts/export_prediction_summaries.py outputs/olmo_test_predictions.jsonl

Wrote summaries to outputs/olmo_test_predictions.txt
